In [0]:
%sql
-- ============================================================================
-- Pipeline Step: Unity Catalog Governance (RLS & CLS)
-- Description: Implements Column Masking (CLS) on sensitive user attributes
--              and Row Filtering (RLS) on fact events.
-- ============================================================================

USE CATALOG dbr_dev;
USE SCHEMA wikimediademo_gold;

-- ----------------------------------------------------------------------------
-- 1. Column-Level Security (CLS): Masking user handles
-- ----------------------------------------------------------------------------
CREATE OR REPLACE FUNCTION mask_user_name(user_name STRING)
RETURN IF(
    is_account_group_member('data_engineers') OR is_account_group_member('admins'),
    user_name,
    concat(substring(user_name, 1, 2), '***')
);

-- Apply column mask to the user column in the Gold fact table
ALTER TABLE dbr_dev.wikimediademo_gold.fact_wikipedia_edits 
ALTER COLUMN user SET MASK mask_user_name;

-- ----------------------------------------------------------------------------
-- 2. Row-Level Security (RLS): Row access filter
-- ----------------------------------------------------------------------------
CREATE OR REPLACE FUNCTION filter_active_edits(editor_type_key INT)
RETURN IF(
    is_account_group_member('admin_group'),
    true,
    editor_type_key IN (1, 2)
);

-- Apply row filter
ALTER TABLE dbr_dev.wikimediademo_gold.fact_wikipedia_edits 
SET ROW FILTER filter_active_edits ON (editor_type_key);

In [0]:
%sql
SELECT 
    event_id,
    user AS raw_or_masked_user,
    -- Direct test of masking function logic:
    mask_user_name(user) AS test_evaluated_mask,
    concat(substring(user, 1, 2), '***') AS expected_anonymous_view
FROM dbr_dev.wikimediademo_gold.fact_wikipedia_edits
LIMIT 5;

In [0]:
%sql
DESCRIBE TABLE EXTENDED dbr_dev.wikimediademo_gold.fact_wikipedia_edits;